# Linear Regression Example

This notebook demonstrates a simple linear regression workflow on synthetic data, including:
- Generating data
- Fitting a model with scikit-learn
- Visualizing the fit and residuals
- Evaluating on a train/test split
- Inspecting statistical summary & confidence / prediction intervals with statsmodels


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set(style="whitegrid")
%matplotlib inline


In [ ]:
# Generate synthetic linear data with noise
np.random.seed(42)
n = 200
X = np.random.uniform(0, 10, n)
true_slope = 2.5
true_intercept = 1.5
noise_sd = 4.0
y = true_slope * X + true_intercept + np.random.normal(0, noise_sd, n)

df = pd.DataFrame({"X": X, "y": y})
df.head()


In [ ]:
# Scatter plot of data and fit a simple linear regression with scikit-learn
X_reshaped = X.reshape(-1, 1)
lr = LinearRegression()
lr.fit(X_reshaped, y)
y_pred = lr.predict(X_reshaped)

print(f"Learned intercept: {lr.intercept_:.3f}")
print(f"Learned slope: {lr.coef_[0]:.3f}")

plt.figure(figsize=(8,6))
sns.scatterplot(x=X, y=y, label="data", alpha=0.7)
# sort for line plotting
order = np.argsort(X)
plt.plot(X[order], y_pred[order], color="red", label="OLS fit (sklearn)")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.title("Scatter plot with linear regression line")
plt.show()


In [ ]:
# Residuals diagnostics
residuals = y - y_pred

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(X, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('X')
plt.ylabel('Residual (y - y_pred)')
plt.title('Residuals vs X')

plt.subplot(1,2,2)
sns.histplot(residuals, kde=True)
plt.title('Residual distribution')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()


In [ ]:
# Train/test split evaluation
X_train, X_test, y_train, y_test = train_test_split(X_reshaped, y, test_size=0.3, random_state=1)
lr2 = LinearRegression()
lr2.fit(X_train, y_train)
y_test_pred = lr2.predict(X_test)

mse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)
print(f"Test MSE: {mse:.3f}")
print(f"Test R^2: {r2:.3f}")

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_test_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual y')
plt.ylabel('Predicted y')
plt.title('Predicted vs Actual (Test set)')
plt.show()


In [ ]:
# Statsmodels OLS for statistical summary and confidence/prediction intervals
model = smf.ols("y ~ X", data=df).fit()
print(model.summary())

# Create a grid of X values to plot the fitted line and intervals
X_grid = np.linspace(df['X'].min(), df['X'].max(), 100)
pred_df = pd.DataFrame({"X": X_grid})
pred = model.get_prediction(pred_df)
pred_summary = pred.summary_frame(alpha=0.05)  # 95% CI
pred_summary.head()

plt.figure(figsize=(8,6))
# scatter data
plt.scatter(df['X'], df['y'], alpha=0.5, label='data')
# fitted mean line
plt.plot(X_grid, pred_summary['mean'], color='red', label='fitted mean')
# 95% confidence interval for the mean
plt.fill_between(X_grid, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], color='red', alpha=0.2, label='95% CI (mean)')
# 95% prediction interval for new observations
plt.fill_between(X_grid, pred_summary['obs_ci_lower'], pred_summary['obs_ci_upper'], color='orange', alpha=0.15, label='95% PI (obs)')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.title('Fit with 95% CI (mean) and 95% PI (obs)')
plt.show()


## Notes and next steps

- This notebook shows a basic single-variable linear regression pipeline. For real datasets, do thorough feature engineering and validation.
- For uncertainty: statsmodels provides t-tests, confidence intervals, and prediction intervals.
- For multivariate regression, pass multiple columns into scikit-learn / statsmodels. Consider regularization (Ridge, Lasso) if features are many or collinear.
- You can extend this notebook to show cross-validation, learning curves, and bootstrap confidence intervals.
